# 02. YOLOv10 Model Training

This notebook handles the full training pipeline for the **Fruit & Vegetable Disease Detection** model using YOLOv10.

**Workflow:**
1. Environment & GPU check
2. Dataset verification
3. Model training with tuned hyperparameters
4. Results visualization (loss curves, mAP, confusion matrix)
5. Export best weights for deployment

## 1. Imports & Path Configuration

In [1]:
import os
import torch
import shutil
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Image, display

# ==========================================
# PATH CONFIGURATION
# ==========================================
PROJECT_ROOT = Path("..").resolve()
DATASET_YAML = PROJECT_ROOT / "data" / "processed" / "dataset.yaml"
PRETRAINED_DIR = PROJECT_ROOT / "models" / "pretrained"
WEIGHTS_DIR = PROJECT_ROOT / "models" / "weights"

# Create directories if they don't exist
PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset YAML : {DATASET_YAML}")
print(f"Pretrained dir: {PRETRAINED_DIR}")
print(f"Weights dir   : {WEIGHTS_DIR}")

Project root : D:\in-class\Advance_AI2\advanced_ai_project
Dataset YAML : D:\in-class\Advance_AI2\advanced_ai_project\data\processed\dataset.yaml
Pretrained dir: D:\in-class\Advance_AI2\advanced_ai_project\models\pretrained
Weights dir   : D:\in-class\Advance_AI2\advanced_ai_project\models\weights


## 2. Environment & GPU Check

In [2]:
print("=" * 50)
print("ENVIRONMENT INFO")
print("=" * 50)
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"GPU device      : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    DEVICE = "cpu"
    print("⚠️  No GPU detected. Training will run on CPU (much slower).")

print(f"Selected device : {DEVICE}")
print("=" * 50)

ENVIRONMENT INFO
PyTorch version : 2.6.0+cu124
CUDA available  : True
GPU device      : NVIDIA GeForce RTX 4070
GPU memory      : 12.0 GB
Selected device : cuda


## 3. Dataset Verification

Verify that the processed dataset from `01_data_annotation.ipynb` exists and has the expected structure.

In [3]:
def verify_dataset(yaml_path: Path) -> bool:
    """Verify that the dataset exists and report split statistics."""
    if not yaml_path.exists():
        print(f"❌ Dataset config not found: {yaml_path}")
        print("   Please run 01_data_annotation.ipynb first.")
        return False

    processed_dir = yaml_path.parent
    print("=" * 50)
    print("DATASET VERIFICATION")
    print("=" * 50)

    total = 0
    for split in ["train", "val", "test"]:
        img_dir = processed_dir / "images" / split
        lbl_dir = processed_dir / "labels" / split

        n_images = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
        n_labels = len(list(lbl_dir.glob("*.txt"))) if lbl_dir.exists() else 0
        total += n_images

        status = "✅" if n_images > 0 and n_images == n_labels else "⚠️"
        print(f"  {status} {split:5s} -> {n_images:5d} images | {n_labels:5d} labels")

    print(f"\n  Total images: {total}")
    print("=" * 50)
    return total > 0

dataset_ok = verify_dataset(DATASET_YAML)
assert dataset_ok, "Dataset is missing or empty. Run 01_data_annotation.ipynb first."

DATASET VERIFICATION
  ⚠️ train -> 23414 images | 22727 labels
  ⚠️ val   ->  4380 images |  4356 labels
  ⚠️ test  ->  1483 images |  1478 labels

  Total images: 29277


## 4. Training Configuration

Define all hyperparameters in one place for easy tuning.

In [4]:
# ==========================================
# TRAINING HYPERPARAMETERS
# ==========================================
YOLO_MODEL = str(PRETRAINED_DIR / "yolov10n.pt")  # Base model: nano variant for faster training
EPOCHS = 50                 # Number of training epochs
IMG_SIZE = 640              # Input image resolution
BATCH_SIZE = 16             # Batch size (reduce to 8 if GPU OOM)
LEARNING_RATE = 0.01        # Initial learning rate
PATIENCE = 10               # Early stopping patience (epochs without improvement)
WORKERS = 4                 # Number of data-loading workers

# Augmentation toggles
USE_MOSAIC = True           # Mosaic augmentation (composite 4 images)
USE_MIXUP = 0.1             # MixUp probability (blend 2 images)
USE_HSVAUG = True           # HSV color-space augmentation

# Training run name (used for output directory)
RUN_NAME = "fruit_disease_v1"

print("=" * 50)
print("TRAINING CONFIGURATION")
print("=" * 50)
print(f"  Model       : {YOLO_MODEL}")
print(f"  Epochs      : {EPOCHS}")
print(f"  Image size  : {IMG_SIZE}")
print(f"  Batch size  : {BATCH_SIZE}")
print(f"  LR          : {LEARNING_RATE}")
print(f"  Patience    : {PATIENCE}")
print(f"  Device      : {DEVICE}")
print(f"  Run name    : {RUN_NAME}")
print("=" * 50)

TRAINING CONFIGURATION
  Model       : yolov10n.pt
  Epochs      : 50
  Image size  : 640
  Batch size  : 16
  LR          : 0.01
  Patience    : 10
  Device      : cuda
  Run name    : fruit_disease_v1


## 5. Load Pretrained Model & Start Training

In [5]:
# Load YOLOv10 pretrained model
print(f"⏳ Loading pretrained model: {YOLO_MODEL} ...")
model = YOLO(YOLO_MODEL)
print("✅ Model loaded successfully.\n")

# Start training
print("🚀 Starting training...\n")
results = model.train(
    data=str(DATASET_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    lr0=LEARNING_RATE,
    patience=PATIENCE,
    workers=WORKERS,
    device=DEVICE,
    project=str(WEIGHTS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    # Augmentation
    mosaic=1.0 if USE_MOSAIC else 0.0,
    mixup=USE_MIXUP,
    hsv_h=0.015 if USE_HSVAUG else 0.0,
    hsv_s=0.7 if USE_HSVAUG else 0.0,
    hsv_v=0.4 if USE_HSVAUG else 0.0,
    # Logging
    verbose=True,
    plots=True,
)

print("\n" + "=" * 50)
print("🎉 TRAINING COMPLETE")
print("=" * 50)

⏳ Loading pretrained model: yolov10n.pt ...
✅ Model loaded successfully.

🚀 Starting training...

New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.0  Python-3.11.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
engine\trainer: task=detect, mode=train, model=yolov10n.pt, data=D:\in-class\Advance_AI2\advanced_ai_project\data\processed\dataset.yaml, epochs=50, time=None, patience=10, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=cuda, workers=4, project=D:\in-class\Advance_AI2\advanced_ai_project\models\weights, name=fruit_disease_v1, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, 

train: Scanning D:\in-class\Advance_AI2\advanced_ai_project\data\processed\labels\train.cache... 23414 images, 0 backgrounds, 0 corrupt: 100%|██████████| 23414/23414 [00:00<?, ?it/s]

train: WARNING  D:\in-class\Advance_AI2\advanced_ai_project\data\processed\images\train\Carrot__Healthy_freshCarrot (3).jpeg: corrupt JPEG restored and saved
train: WARNING  D:\in-class\Advance_AI2\advanced_ai_project\data\processed\images\train\Carrot__Healthy_freshCarrot (475).jpg: corrupt JPEG restored and saved
train: WARNING  D:\in-class\Advance_AI2\advanced_ai_project\data\processed\images\train\Tomato__Healthy_freshTomato (122).jpg: corrupt JPEG restored and saved



val: Scanning D:\in-class\Advance_AI2\advanced_ai_project\data\processed\labels\val.cache... 4380 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4380/4380 [00:00<?, ?it/s]

val: WARNING  D:\in-class\Advance_AI2\advanced_ai_project\data\processed\images\val\Carrot__Healthy_freshCarrot (491).jpg: corrupt JPEG restored and saved


...

      50/50      3.02G     0.7852      1.175      2.125         13        640: 100%|██████████| 1464/1464 [02:43<00:00,  8.97it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 137/137 [00:17<00:00,  7.95it/s]

                   all       4380       7702      0.759      0.721      0.758      0.698



50 epochs completed in 2.610 hours.
Optimizer stripped from D:\in-class\Advance_AI2\advanced_ai_project\models\weights\fruit_disease_v1\weights\last.pt, 5.8MB
Optimizer stripped from D:\in-class\Advance_AI2\advanced_ai_project\models\weights\fruit_disease_v1\weights\best.pt, 5.8MB

Validating D:\in-class\Advance_AI2\advanced_ai_project\models\weights\fruit_disease_v1\weights\best.pt...
Ultralytics 8.3.0  Python-3.11.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
YOLOv10n summary (fused): 285 layers, 2,705,336 parameters, 0 gradients, 8.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 137/137 [00:15<00:00,  8.63it/s]


                   all       4380       7702      0.758      0.722      0.758      0.697
        Apple__Healthy        365        694      0.881      0.852      0.903      0.861
         Apple__Rotten        438        592      0.935       0.78       0.84      0.767
       Banana__Healthy        299        597      0.791       0.73       0.82      0.667
        Banana__Rotten        419        689      0.736      0.733      0.799      0.658
   Bellpepper__Healthy         91        239      0.623      0.665      0.705      0.651
    Bellpepper__Rotten         88        130      0.593      0.608      0.581      0.504
       Carrot__Healthy         92        523      0.655      0.707       0.72      0.632
        Carrot__Rotten         86        231      0.561      0.487      0.526      0.436
     Cucumber__Healthy         91        127      0.738      0.732      0.799      0.722
      Cucumber__Rotten         88        102      0.793       0.75      0.831      0.749
        Grape__Health

## 6. Results Visualization

Display the training plots that YOLO automatically generates.

In [ ]:
# Path to the training run output directory
RUN_DIR = WEIGHTS_DIR / RUN_NAME

def show_plot(filename: str, title: str):
    """Display a training result image if it exists."""
    path = RUN_DIR / filename
    if path.exists():
        print(f"\n{'─' * 40}")
        print(f"📊 {title}")
        print(f"{'─' * 40}")
        display(Image(filename=str(path), width=800))
    else:
        print(f"⚠️  {title} not found: {path.name}")

# -- Loss & mAP curves --
show_plot("results.png", "Training & Validation Loss / mAP Curves")

# -- Confusion Matrix --
show_plot("confusion_matrix_normalized.png", "Normalized Confusion Matrix")
show_plot("confusion_matrix.png", "Confusion Matrix (raw counts)")

# -- PR Curve --
show_plot("PR_curve.png", "Precision-Recall Curve")

# -- F1 Curve --
show_plot("F1_curve.png", "F1-Confidence Curve")

# -- Sample predictions on validation set --
show_plot("val_batch0_pred.png", "Validation Batch 0 — Predictions")
show_plot("val_batch0_labels.png", "Validation Batch 0 — Ground Truth")

## 7. Evaluate on Test Set

In [ ]:
# Load the best checkpoint from training
best_weights = RUN_DIR / "weights" / "best.pt"
assert best_weights.exists(), f"Best weights not found at {best_weights}"

print(f"⏳ Loading best weights: {best_weights}")
best_model = YOLO(str(best_weights))

# Run evaluation on the test split
print("\n🔍 Evaluating on the TEST set...\n")
test_metrics = best_model.val(
    data=str(DATASET_YAML),
    split="test",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    verbose=True,
)

print("\n" + "=" * 50)
print("TEST SET RESULTS")
print("=" * 50)
print(f"  mAP@50      : {test_metrics.box.map50:.4f}")
print(f"  mAP@50-95   : {test_metrics.box.map:.4f}")
print(f"  Precision   : {test_metrics.box.mp:.4f}")
print(f"  Recall      : {test_metrics.box.mr:.4f}")
print("=" * 50)

## 8. Export Best Weights

Copy the best checkpoint to the project's standard weights directory for easy access by other modules (`src/detection/detector.py`).

In [ ]:
# Copy best.pt to the top-level weights directory
export_path = WEIGHTS_DIR / "best.pt"
shutil.copy(best_weights, export_path)
print(f"✅ Best weights exported to: {export_path}")

# Also copy last.pt as a backup
last_weights = RUN_DIR / "weights" / "last.pt"
if last_weights.exists():
    shutil.copy(last_weights, WEIGHTS_DIR / "last.pt")
    print(f"✅ Last weights exported to: {WEIGHTS_DIR / 'last.pt'}")

print("\n" + "=" * 50)
print("🎉 ALL DONE — Model is ready for inference!")
print(f"   Use weights at: {export_path}")
print("=" * 50)